In [1]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../')

In [2]:
import torch

import pandas as pd

In [3]:
data = pd.read_csv('data/tweets.csv')
data.rename(columns={'tweet_text': 'text'}, inplace=True)
data.drop_duplicates(subset=['text'], inplace=True)
data = data.reset_index(drop=True)
label = 'cyberbullying_type'
text = 'text'
text_prepared = 'text_prepared'

In [30]:
try:
    import importlib, sys
    importlib.reload(sys.modules['src.dataset'])
except:
    pass
from src.dataset import Dataset

In [31]:
ds = Dataset()
ds.texts = ds.texts[:int(len(ds.texts) * 0.4)]
ds.build_pairs()
# a.texts = ["annie, worried, blondes, late, dummy, mkr, sheesh".split(",")]

In [32]:
from torch.utils.data import DataLoader
dl = DataLoader(ds, batch_size=128)

In [33]:
try:
    import importlib, sys
    importlib.reload(sys.modules['src.model'])
except:
    pass
from src.model import Word2Vec

In [34]:
w2i, i2w = ds.get_w2i_i2w()
w2v = Word2Vec(word_size=len(w2i), hidden_size=256, w2i=w2i, i2w=i2w)

In [35]:
import torch

In [36]:
device = 'cuda' if torch.cuda.is_available() else 'cpue'

In [57]:
def eval(model, dl):
    model.eval()
    all_samples = 0
    eq = 0
    for x, y in tqdm.tqdm(dl):
        import torch.nn.functional as F
        y = torch.tensor([w2i[i] for i in y])
        y = F.one_hot(y, num_classes=len(w2i)).float()
        y = y.to(device)
        optimizer.zero_grad()
        preds = w2v(x).argmax(dim=1)
        all_samples += len(x)
        eq += (preds == y.argmax(dim=1)).sum()
    model.train()
    return eq / all_samples

In [60]:
from torch.optim import AdamW
import torch.nn as nn
import tqdm

w2v = w2v.to(device)

optimizer = AdamW(w2v.parameters())
criteria = nn.CrossEntropyLoss()

dl = dl

n_epoch = 20

losses = []
for epoch in range(n_epoch):
    cur_loss = 0
    for x, y in tqdm.tqdm(dl):
        import torch.nn.functional as F
        y = torch.tensor([w2i[i] for i in y])
        y = F.one_hot(y, num_classes=len(w2i)).float()
        y = y.to(device)
        optimizer.zero_grad()
        preds = w2v(x)
        loss = criteria(preds, y)
        loss.backward()
        optimizer.step()
        cur_loss += loss.item()
    losses.append([cur_loss / len(dl), eval(w2v, dl)])
    print(f"loss: {losses[-1][0]}, accuracy: {losses[-1][1]}")
        

100%|██████████| 2500/2500 [00:12<00:00, 193.23it/s]


loss: 7.872431104516983, accuracy: 0.03126846253871918


100%|██████████| 2500/2500 [00:12<00:00, 196.47it/s]


loss: 5.56255550647378, accuracy: 0.0438077375292778


100%|██████████| 2500/2500 [00:12<00:00, 197.18it/s]


loss: 4.980171311175823, accuracy: 0.04738413542509079


... (30 outputs left out)

100%|██████████| 2500/2500 [00:12<00:00, 200.34it/s]


loss: 4.1026242639064785, accuracy: 0.04704025015234947


100%|██████████| 2500/2500 [00:12<00:00, 198.66it/s]

loss: 4.0933458196401595, accuracy: 0.04716529697179794


In [58]:
eval(w2v, dl)

100%|██████████| 2500/2500 [00:12<00:00, 200.10it/s]


tensor(0.0454, device='cuda:0')

11004

In [100]:
word_vector = w2v.A.weight.T[w2i['bully'], :]
ans = ((w2v.A.weight.T @ word_vector))

In [101]:
values, indices = torch.topk(ans.squeeze(), k=5)

print(values)
print([i2w[i] for i in indices.tolist()])

tensor([74.2049, 19.4678, 18.2531, 16.9276, 16.8737], device='cuda:0',
       grad_fn=<TopkBackward0>)
['bully', 'disekolah', 'baju', 'pake', 'sama']
